# Audio Emotion Recognition — Fine-tuning Wav2Vec2 on RAVDESS

Fine-tunes `facebook/wav2vec2-base` on the `xbgoose/ravdess` dataset for emotion classification.

> **Note:** RAVDESS assigns exactly one emotion per audio clip, so this is **multi-class**
> (single-label) classification over 8 emotion categories.

### 8 RAVDESS emotion classes
| ID | Emotion   |
|----|----------|
| 0  | neutral   |
| 1  | calm      |
| 2  | happy     |
| 3  | sad       |
| 4  | angry     |
| 5  | fearful   |
| 6  | disgust   |
| 7  | surprised |

### Setup
Run `uv sync` in the project root before launching Jupyter, then select this project's kernel.

In [ ]:
# Quick sanity check
try:
    import transformers, datasets, torch, sklearn, evaluate
    print(f'transformers : {transformers.__version__}')
    print(f'datasets     : {datasets.__version__}')
    print(f'torch        : {torch.__version__}')
    print(f'CUDA         : {torch.cuda.is_available()} '
          f'({torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU only"})')
except ImportError as e:
    print(f'[ERROR] Missing package: {e}')
    print('Run `uv sync` in the project root, then restart the kernel.')

In [ ]:
import os
import warnings
warnings.filterwarnings('ignore')

import numpy as np
import torch
import matplotlib.pyplot as plt
import seaborn as sns

from datasets import load_dataset, Audio
from transformers import (
    Wav2Vec2FeatureExtractor,
    Wav2Vec2ForSequenceClassification,
    TrainingArguments,
    Trainer,
)
from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    classification_report,
    f1_score,
    confusion_matrix,
)

print('All imports successful.')

In [ ]:
# ── Configuration ────────────────────────────────────────────────────────────
MODEL_NAME    = 'facebook/wav2vec2-base'
DATASET_NAME  = 'xbgoose/ravdess'

SAMPLING_RATE = 16_000          # wav2vec2 expects 16 kHz
MAX_DURATION  = 5.0             # seconds; clips longer than this are truncated
MAX_LENGTH    = int(SAMPLING_RATE * MAX_DURATION)   # 80 000 samples

TRAIN_RATIO   = 0.70
VAL_RATIO     = 0.15
TEST_RATIO    = 0.15

BATCH_SIZE    = 8
NUM_EPOCHS    = 10
LEARNING_RATE = 3e-5
WEIGHT_DECAY  = 0.01
WARMUP_RATIO  = 0.1
OUTPUT_DIR    = './wav2vec2-ravdess-output'
SEED          = 42

device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'Device: {device}')
print(f'Max audio length: {MAX_LENGTH:,} samples ({MAX_DURATION}s @ {SAMPLING_RATE} Hz)')

## 1. Load & Explore Dataset

In [ ]:
raw_dataset = load_dataset(DATASET_NAME)
print(raw_dataset)

In [ ]:
# Use whichever split is available (some datasets ship only 'train')
base_split = list(raw_dataset.keys())[0]
example    = raw_dataset[base_split][0]

print('Available splits :', list(raw_dataset.keys()))
print('Columns          :', list(example.keys()))
print('Audio info       :', {k: v for k, v in example['audio'].items() if k != 'array'})

# Detect the label column
label_col = next(
    (c for c in ['label', 'emotion', 'labels', 'target'] if c in example),
    None
)
assert label_col is not None, 'Could not detect label column — check dataset columns above.'
print(f'Label column     : "{label_col}"')

# Class distribution
all_labels = raw_dataset[base_split][label_col]
unique, counts = np.unique(all_labels, return_counts=True)
print(f'Unique labels : {unique}')
print(f'Total samples : {len(all_labels):,}')
for u, c in zip(unique, counts):
    print(f'  {u:>3} -> {c} samples')

## 2. Define Emotion Labels

In [ ]:
# Standard RAVDESS emotion names in canonical order (0-indexed)
RAVDESS_NAMES = ['neutral', 'calm', 'happy', 'sad', 'angry', 'fearful', 'disgust', 'surprised']

features = raw_dataset[base_split].features
if hasattr(features.get(label_col), 'names'):
    label_names = features[label_col].names
    print('Using dataset ClassLabel names:', label_names)
else:
    min_label = int(min(all_labels))
    n_labels  = int(max(all_labels)) - min_label + 1
    label_names = RAVDESS_NAMES[:n_labels]
    if min_label != 0:
        print(f'Labels start at {min_label}; will shift to 0-indexed during preprocessing.')
    print('Using default RAVDESS names:', label_names)

id2label   = {i: name for i, name in enumerate(label_names)}
label2id   = {name: i for i, name in enumerate(label_names)}
NUM_LABELS = len(label_names)

print(f'id2label   : {id2label}')
print(f'Num classes: {NUM_LABELS}')

## 3. Train / Validation / Test Split

Stratified split preserving class proportions: **70% train / 15% val / 15% test**.

In [ ]:
base_data   = raw_dataset[base_split]
base_labels = base_data[label_col]
indices     = list(range(len(base_data)))

train_idx, temp_idx = train_test_split(
    indices,
    test_size=(VAL_RATIO + TEST_RATIO),
    stratify=base_labels,
    random_state=SEED,
)
temp_labels = [base_labels[i] for i in temp_idx]
val_idx, test_idx = train_test_split(
    temp_idx,
    test_size=TEST_RATIO / (VAL_RATIO + TEST_RATIO),
    stratify=temp_labels,
    random_state=SEED,
)

train_raw = base_data.select(train_idx)
val_raw   = base_data.select(val_idx)
test_raw  = base_data.select(test_idx)

print(f'Train : {len(train_raw):>5,} samples  ({len(train_raw)/len(base_data)*100:.1f}%)')
print(f'Val   : {len(val_raw):>5,} samples  ({len(val_raw)/len(base_data)*100:.1f}%)')
print(f'Test  : {len(test_raw):>5,} samples  ({len(test_raw)/len(base_data)*100:.1f}%)')

## 4. Feature Extraction & Preprocessing

- Resample all audio to **16 kHz** (wav2vec2 requirement).
- Normalize waveforms with `Wav2Vec2FeatureExtractor`.
- Truncate to `MAX_LENGTH` samples; pad shorter clips to the same fixed length.

In [ ]:
feature_extractor = Wav2Vec2FeatureExtractor.from_pretrained(MODEL_NAME)
print('Feature extractor sampling rate:', feature_extractor.sampling_rate)

# Cast audio column so HuggingFace resamples on decode
train_raw = train_raw.cast_column('audio', Audio(sampling_rate=SAMPLING_RATE))
val_raw   = val_raw.cast_column(  'audio', Audio(sampling_rate=SAMPLING_RATE))
test_raw  = test_raw.cast_column( 'audio', Audio(sampling_rate=SAMPLING_RATE))

_min_label = int(min(all_labels))  # shift if labels start at 1

def preprocess(batch):
    audio_arrays = [x['array'] for x in batch['audio']]
    inputs = feature_extractor(
        audio_arrays,
        sampling_rate=SAMPLING_RATE,
        max_length=MAX_LENGTH,
        truncation=True,
        padding='max_length',
        return_attention_mask=True,
    )
    inputs['labels'] = [int(l) - _min_label for l in batch[label_col]]
    return inputs

In [ ]:
cols_to_remove = train_raw.column_names

train_ds = train_raw.map(preprocess, batched=True, remove_columns=cols_to_remove, desc='Preprocessing train')
val_ds   = val_raw.map(  preprocess, batched=True, remove_columns=cols_to_remove, desc='Preprocessing val  ')
test_ds  = test_raw.map( preprocess, batched=True, remove_columns=cols_to_remove, desc='Preprocessing test ')

train_ds.set_format('torch')
val_ds.set_format(  'torch')
test_ds.set_format( 'torch')

print('Columns after preprocessing:', train_ds.column_names)
print('input_values shape (first sample):', train_ds[0]['input_values'].shape)
print('Sample label:', train_ds[0]['labels'].item(), '->', id2label[train_ds[0]['labels'].item()])

## 5. Model Initialization

Load `Wav2Vec2ForSequenceClassification` with a fresh classification head.
The transformer encoder weights come from the pretrained checkpoint.

In [ ]:
model = Wav2Vec2ForSequenceClassification.from_pretrained(
    MODEL_NAME,
    num_labels=NUM_LABELS,
    id2label=id2label,
    label2id=label2id,
    ignore_mismatched_sizes=True,
)

total     = sum(p.numel() for p in model.parameters())
trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f'Total parameters    : {total:,}')
print(f'Trainable parameters: {trainable:,}')

## 6. Training

Uses HuggingFace `Trainer` with per-epoch evaluation on the validation set.
Best checkpoint is selected by `eval_f1_macro`.

In [ ]:
def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=-1)
    return {
        'f1_macro'   : f1_score(labels, preds, average='macro',    zero_division=0),
        'f1_weighted': f1_score(labels, preds, average='weighted', zero_division=0),
        'accuracy'   : float((preds == labels).mean()),
    }

In [ ]:
training_args = TrainingArguments(
    output_dir=OUTPUT_DIR,
    num_train_epochs=NUM_EPOCHS,
    per_device_train_batch_size=BATCH_SIZE,
    per_device_eval_batch_size=BATCH_SIZE,
    eval_strategy='epoch',
    save_strategy='epoch',
    load_best_model_at_end=True,
    metric_for_best_model='f1_macro',
    greater_is_better=True,
    learning_rate=LEARNING_RATE,
    warmup_ratio=WARMUP_RATIO,
    weight_decay=WEIGHT_DECAY,
    logging_steps=10,
    fp16=torch.cuda.is_available(),
    dataloader_num_workers=2,
    report_to='none',
    seed=SEED,
)

In [ ]:
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_ds,
    eval_dataset=val_ds,
    compute_metrics=compute_metrics,
)

train_result = trainer.train()
print('\nTraining complete.')
print(f'Runtime  : {train_result.metrics["train_runtime"]:.1f}s')
print(f'Samples/s: {train_result.metrics["train_samples_per_second"]:.2f}')

## 7. Test-Set Evaluation

Runs inference on the held-out test set and reports:
- Per-class classification report
- F1-Macro score
- Confusion matrix heatmap

In [ ]:
test_output = trainer.predict(test_ds)
test_preds  = np.argmax(test_output.predictions, axis=-1)
test_labels = test_output.label_ids

label_names = [id2label[i] for i in range(NUM_LABELS)]
print(f'Test samples: {len(test_labels):,}')

In [ ]:
print('=' * 64)
print('CLASSIFICATION REPORT')
print('=' * 64)
print(classification_report(test_labels, test_preds, target_names=label_names, zero_division=0))

In [ ]:
f1_macro    = f1_score(test_labels, test_preds, average='macro',    zero_division=0)
f1_weighted = f1_score(test_labels, test_preds, average='weighted', zero_division=0)
accuracy    = float((test_preds == test_labels).mean())

print(f'F1-Macro    : {f1_macro:.4f}')
print(f'F1-Weighted : {f1_weighted:.4f}')
print(f'Accuracy    : {accuracy:.4f}')

In [ ]:
cm = confusion_matrix(test_labels, test_preds)

fig, ax = plt.subplots(figsize=(10, 8))
sns.heatmap(
    cm,
    annot=True,
    fmt='d',
    cmap='Blues',
    xticklabels=label_names,
    yticklabels=label_names,
    ax=ax,
    linewidths=0.5,
)
ax.set_title(f'Confusion Matrix — Test Set  (F1-Macro={f1_macro:.3f})', fontsize=14, pad=14)
ax.set_ylabel('True Label', fontsize=12)
ax.set_xlabel('Predicted Label', fontsize=12)
plt.xticks(rotation=45, ha='right')
plt.tight_layout()
plt.savefig('confusion_matrix.png', dpi=150)
plt.show()
print('Saved -> confusion_matrix.png')

## 8. Training Loss Curve

Training loss (per logging step) and validation loss + F1-Macro (per epoch).

In [ ]:
log_history = trainer.state.log_history

train_steps  = [e['step'] for e in log_history if 'loss' in e and 'eval_loss' not in e]
train_losses = [e['loss'] for e in log_history if 'loss' in e and 'eval_loss' not in e]
eval_epochs  = [e['epoch']     for e in log_history if 'eval_loss' in e]
eval_losses  = [e['eval_loss'] for e in log_history if 'eval_loss' in e]
eval_f1      = [e.get('eval_f1_macro') for e in log_history if 'eval_loss' in e]

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].plot(train_steps, train_losses, color='steelblue', alpha=0.85, linewidth=1.2)
axes[0].set_xlabel('Step')
axes[0].set_ylabel('Cross-Entropy Loss')
axes[0].set_title('Training Loss')
axes[0].grid(True, alpha=0.3)

color_loss, color_f1 = 'coral', 'seagreen'
ax_loss = axes[1]
ax_loss.plot(eval_epochs, eval_losses, color=color_loss, marker='o', label='Val Loss')
ax_loss.set_xlabel('Epoch')
ax_loss.set_ylabel('Loss', color=color_loss)
ax_loss.tick_params(axis='y', labelcolor=color_loss)
ax_loss.set_title('Validation Loss & F1-Macro per Epoch')

if any(v is not None for v in eval_f1):
    ax_f1 = ax_loss.twinx()
    ax_f1.plot(eval_epochs, eval_f1, color=color_f1, marker='s', linestyle='--', label='Val F1-Macro')
    ax_f1.set_ylabel('F1-Macro', color=color_f1)
    ax_f1.tick_params(axis='y', labelcolor=color_f1)
    lines  = ax_loss.get_lines() + ax_f1.get_lines()
    labels = [l.get_label() for l in lines]
    ax_loss.legend(lines, labels, loc='upper right')
else:
    ax_loss.legend(loc='upper right')

ax_loss.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig('training_curves.png', dpi=150)
plt.show()
print('Saved -> training_curves.png')